# 🎓 Socratic Science Tutor — Dual T4 GPU Fine-Tuning

### ⚡ Master Features:
- **Dataset**: `/kaggle/input/datasets/shekhu1/socratic-final/socratic_dataset.jsonl` (auto-synced to `Susu11/socratic_idea_expansion`)
- **Base Model**: `Qwen/Qwen2.5-7B-Instruct`
- **Target Model Output**: [`Susu11/socratic_qwen8b`](https://huggingface.co/Susu11/socratic_qwen8b)
- **Per-Epoch Upload**: Automatically saves and pushes checkpoints to Hugging Face after **each epoch**!
- **GGUF Export**: Converts merged model to `.gguf` (for Ollama / llama.cpp / LM Studio) and uploads directly to Hugging Face!

---

## 1. Install Dependencies

In [ ]:
!pip install -q transformers peft trl datasets accelerate bitsandbytes wandb huggingface_hub gguf
print("✅ All dependencies installed successfully!")

## 2. Verify Dual T4 GPUs

In [ ]:
import torch

gpu_count = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Total GPUs detected: {gpu_count}")
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  -> GPU {i}: {name} ({mem:.2f} GB VRAM)")

assert gpu_count >= 1, "⚠️ No GPU detected! Go to Notebook Settings -> Accelerator -> select GPU T4 x2."

## 3. Automatic Authentication (HF_TOKEN & WANDB_TOKEN)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

def clean(v):
    return v.strip().strip("'").strip('"').strip() if v else None

secrets = UserSecretsClient()

# 1. Read HF_TOKEN
hf_token = None
for label in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "huggingface", "HF"]:
    try:
        hf_token = clean(secrets.get_secret(label))
        if hf_token: break
    except Exception:
        pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    try:
        login(token=hf_token, add_to_git_credential=True)
        print("✅ Hugging Face authentication successful!")
    except Exception as e:
        print(f"⚠️ HF login notice: {e}")
else:
    print("⚠️ HF_TOKEN not found in Kaggle Secrets.")

# 2. Read WANDB_TOKEN
wandb_token = None
for label in ["WANDB_TOKEN", "WANDB_API_KEY", "wandb_token", "wandb_key", "wandb"]:
    try:
        wandb_token = clean(secrets.get_secret(label))
        if wandb_token: break
    except Exception:
        pass

if wandb_token:
    os.environ["WANDB_API_KEY"] = wandb_token
    os.environ["WANDB_PROJECT"] = "socratic-model-fine-tune"
    try:
        wandb.login(key=wandb_token)
        print("✅ Weights & Biases authentication successful!")
    except Exception as e:
        print(f"ℹ️ Wandb login notice: {e}")
        os.environ["WANDB_DISABLED"] = "true"
else:
    print("ℹ️ WANDB_TOKEN not found. Running in offline mode.")
    os.environ["WANDB_DISABLED"] = "true"

## 4. Load Dataset from Kaggle Input & Sync to Hugging Face
Reads directly from `/kaggle/input/datasets/shekhu1/socratic-final/socratic_dataset.jsonl`.

In [ ]:
import os
import json
from datasets import Dataset, load_dataset

DATASET_REPO = "Susu11/socratic_idea_expansion"

candidate_paths = [
    "/kaggle/input/datasets/shekhu1/socratic-final/socratic_dataset.jsonl",
    "/kaggle/input/socratic-final/socratic_dataset.jsonl",
    "socratic_dataset.jsonl"
]

target_file = None
for path in candidate_paths:
    if os.path.exists(path):
        target_file = path
        break

if target_file:
    print(f"📂 Found dataset file at: {target_file}")
    records = [json.loads(line) for line in open(target_file, "r", encoding="utf-8") if line.strip()]
    dataset = Dataset.from_list(records)
    print(f"✅ Successfully loaded {len(dataset)} conversation examples from disk!")
    
    if os.getenv("HF_TOKEN"):
        try:
            print(f"📤 Syncing to Hugging Face Hub: https://huggingface.co/datasets/{DATASET_REPO}...")
            dataset.push_to_hub(DATASET_REPO, private=False)
            print("🎉 Dataset synced to Hugging Face Hub successfully!")
        except Exception as e:
            print(f"ℹ️ Hub upload notice: {e}")
else:
    print(f"🌐 Local file not found, loading directly from Hub: {DATASET_REPO}...")
    dataset = load_dataset(DATASET_REPO, split="train")
    print(f"✅ Loaded {len(dataset)} examples from Hugging Face Hub!")

print("\n--- First Dialogue Sample ---")
for msg in dataset[0]["messages"]:
    print(f"[{msg['role'].upper()}]:\n{msg['content']}\n")

## 5. Load Qwen2.5-7B-Instruct with 4-bit QLoRA across Dual T4 GPUs (Strict GPU Allocation)

In [ ]:
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Clean previous VRAM allocations to prevent CPU offload
try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"🤖 Base Model: {BASE_MODEL_ID}")

# 2. Tokenizer and Chat Template
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def apply_chat_template(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
            for conv in batch["messages"]
        ]
    }

train_dataset = dataset.map(apply_chat_template, batched=True)
print("📋 Formatted chat text preview:")
print(train_dataset[0]["text"][:350] + "...\n")

# 3. QLoRA 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 4. Explicit max_memory allocation: Restrict exclusively to GPUs so it NEVER spills to CPU
num_gpus = torch.cuda.device_count()
max_memory = {i: "14GiB" for i in range(num_gpus)}
print(f"Target GPU Memory Allocation: {max_memory}")

# 5. Load Model across GPUs
print("⏳ Loading model across both T4 GPUs (takes ~1-2 mins)... ")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,  # Guarantees everything stays strictly inside GPU VRAM
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

# 6. LoRA Adapter
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
print("\n📊 Trainable parameters:")
model.print_trainable_parameters()

## 6. Train Socratic Model with SFTTrainer (Adaptive Parameter Compatibility)
- Dynamically adapts to any TRL version (v0.11, v0.12, v0.13, v0.14+).
- Uploads checkpoints to Hugging Face after **each epoch**!

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./socratic_tutor_lora"
HUB_MODEL_ID = "Susu11/socratic_qwen8b"

# Base training arguments
base_args = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_steps": 10,
    "lr_scheduler_type": "cosine",
    "logging_steps": 5,
    "save_strategy": "epoch",
    "push_to_hub": bool(os.getenv("HF_TOKEN")),
    "hub_model_id": HUB_MODEL_ID if os.getenv("HF_TOKEN") else None,
    "hub_strategy": "every_save",
    "fp16": True,
    "bf16": False,
    "max_grad_norm": 0.3,
    "optim": "paged_adamw_8bit",
    "report_to": ["wandb"] if os.getenv("WANDB_API_KEY") else ["none"],
    "run_name": "kaggle-dual-t4-qwen-run",
}

# Adaptively filter parameters for SFTConfig
sft_config_params = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in sft_config_params:
    base_args["max_length"] = 2048
elif "max_seq_length" in sft_config_params:
    base_args["max_seq_length"] = 2048

if "dataset_text_field" in sft_config_params:
    base_args["dataset_text_field"] = "text"

valid_config_args = {k: v for k, v in base_args.items() if k in sft_config_params}
training_args = SFTConfig(**valid_config_args)

# Adaptively configure SFTTrainer
sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "peft_config": peft_config,
    "args": training_args,
}

if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

if "dataset_text_field" in sft_trainer_params and "dataset_text_field" not in valid_config_args:
    trainer_kwargs["dataset_text_field"] = "text"

if "max_seq_length" in sft_trainer_params and "max_seq_length" not in valid_config_args and "max_length" not in valid_config_args:
    trainer_kwargs["max_seq_length"] = 2048

trainer = SFTTrainer(**trainer_kwargs)

print("🚀 Starting Training on Dual T4 GPUs (syncing to HF at every epoch)... ")
train_result = trainer.train()
print("✅ Training Completed Successfully!")
print(f"📉 Final Training Loss: {train_result.training_loss:.4f}")

## 7. Live Test: Verify Socratic Persona & `<plan>` Tags

In [ ]:
system_prompt = (
    "You are a Socratic Science Tutor for a Grade 10 student. "
    "Your goal is to guide the student to discover concepts through reasoning, NEVER by giving the final answer directly. "
    "Ask EXACTLY ONE question per turn. Keep responses to 1-3 sentences. "
    "If the student is stuck, provide a simpler analogy or break the concept into a smaller step. "
    "State your pedagogical goal inside <plan>...</plan> tags."
)

test_question = "Why do stars twinkle in the night sky, but planets don't?"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_question}
]

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n--- Generated Socratic Response ---")
print(generated_response)

## 8. Merge LoRA into 16-bit Standalone Model & Push to Hugging Face

In [ ]:
import gc
from peft import PeftModel

MERGED_DIR = "./socratic_tutor_merged"

# 1. Save Adapter Locally & Push Final
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
if os.getenv("HF_TOKEN"):
    trainer.model.push_to_hub(HUB_MODEL_ID)
    tokenizer.push_to_hub(HUB_MODEL_ID)

# 2. Free VRAM
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

# 3. Merge Adapter into Base Model
print("Reloading base model in FP16 to merge LoRA weights...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged_model = merged_model.merge_and_unload()

os.makedirs(MERGED_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"💾 Standalone 16-bit model saved locally to: {MERGED_DIR}")

# 4. Push Merged 16-bit Model to Hub
if os.getenv("HF_TOKEN"):
    merged_hub_id = f"{HUB_MODEL_ID}-merged"
    print(f"📤 Pushing full merged 16-bit model to: https://huggingface.co/{merged_hub_id}...")
    try:
        merged_model.push_to_hub(merged_hub_id)
        tokenizer.push_to_hub(merged_hub_id)
        print(f"🎉 Merged model successfully uploaded to: https://huggingface.co/{merged_hub_id}")
    except Exception as e:
        print(f"⚠️ Error pushing merged model: {e}")

## 9. 📦 Convert to GGUF Format (for Ollama, LM Studio, llama.cpp) & Upload to Hub
Converts your trained model to a `.gguf` file and uploads it to [`Susu11/socratic_qwen8b`](https://huggingface.co/Susu11/socratic_qwen8b).

In [ ]:
import subprocess
import sys
from huggingface_hub import HfApi

GGUF_OUT = "socratic_qwen_f16.gguf"
MERGED_DIR = "./socratic_tutor_merged"

print("📦 Step 1: Setting up llama.cpp for GGUF conversion...")
if not os.path.exists("llama.cpp"):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
    !pip install -q -r llama.cpp/requirements.txt

print(f"📦 Step 2: Converting '{MERGED_DIR}' to '{GGUF_OUT}'...")
subprocess.run([
    sys.executable,
    "llama.cpp/convert_hf_to_gguf.py",
    MERGED_DIR,
    "--outfile", GGUF_OUT,
    "--outtype", "f16"
], check=True)

file_size_gb = os.path.getsize(GGUF_OUT) / 1e9
print(f"✅ GGUF file generated: {GGUF_OUT} ({file_size_gb:.2f} GB)")

# Step 3: Upload GGUF to Hugging Face Hub
if os.getenv("HF_TOKEN"):
    print(f"📤 Uploading GGUF to Hugging Face Hub: {HUB_MODEL_ID}...")
    api = HfApi()
    try:
        api.upload_file(
            path_or_fileobj=GGUF_OUT,
            path_in_repo=GGUF_OUT,
            repo_id=HUB_MODEL_ID,
            repo_type="model",
        )
        print(f"🎉 GGUF file successfully uploaded to: https://huggingface.co/{HUB_MODEL_ID}")
    except Exception as e:
        print(f"⚠️ Error uploading GGUF file: {e}")
else:
    print("ℹ️ HF_TOKEN not set; GGUF file saved locally in /kaggle/working/.")